In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy
import mlxtend

In [30]:
df = pd.read_csv("data/ecology/environmental_field_reports.csv")
df.head()

,document_id,site_code,document_date,document_type,author_name,author_organization,reporting_period,title,text,related_survey_id,related_device_reference,publication_status
0,EFR-510088,KNT-KES,2024-04-02,ecological_incident_report,Anika Dren,Horizon Field Research Cooperative,2024-04,Post-Access Ecological Incident at KNT-KES,INCIDENT\r\nA post-access inspection was condu...,BAT-0000064,DEV-KNTKES-2,restricted
1,EFR-510082,KNT-JAK,2019-04-01,field_observation,Vera Waren,Pollinator Justice Network,2019-04,Field Observation 27,SITE AND TIME\r\nField observation at KNT-JAK ...,BAT-0001000,DEV-KNTJAK-3,restricted
2,EFR-510035,KNT-JQG,2022-09-08,field_observation,Uma Wessen,Horizon Field Research Cooperative,2022-09,Field Observation 28,SITE AND TIME\r\nField observation at KNT-JQG ...,BAT-0001037,DEV-KNTJQG-1,internal
3,EFR-510007,KNT-JQG,2025-02-15,field_observation,J. Varlan,Varlan Natural History Circle,2025-02,Field Observation 29,SITE AND TIME\r\nField observation at KNT-JQG ...,BAT-0001074,DEV-KNTJQG-1,public
4,EFR-510031,KNT-VLB,2020-07-22,field_observation,Jonah Bell,Bell Conservation Survey,2020-07,Field Observation 30,SITE AND TIME\r\nField observation at KNT-VLB ...,BAT-0001111,DEV-KNTVLB-2,internal


## Vérification de la qualité des données

In [31]:
df.head()
print("Dimensions :", df.shape)
print("\nTypes des colonnes :")
print(df.dtypes)
display(df.head())
display(df.sample(min(5, len(df)), random_state=42))

Dimensions : (30, 12)

Types des colonnes :
document_id                 str
site_code                   str
document_date               str
document_type               str
author_name                 str
author_organization         str
reporting_period            str
title                       str
text                        str
related_survey_id           str
related_device_reference    str
publication_status          str
dtype: object


,document_id,site_code,document_date,document_type,author_name,author_organization,reporting_period,title,text,related_survey_id,related_device_reference,publication_status
0,EFR-510088,KNT-KES,2024-04-02,ecological_incident_report,Anika Dren,Horizon Field Research Cooperative,2024-04,Post-Access Ecological Incident at KNT-KES,INCIDENT\r\nA post-access inspection was condu...,BAT-0000064,DEV-KNTKES-2,restricted
1,EFR-510082,KNT-JAK,2019-04-01,field_observation,Vera Waren,Pollinator Justice Network,2019-04,Field Observation 27,SITE AND TIME\r\nField observation at KNT-JAK ...,BAT-0001000,DEV-KNTJAK-3,restricted
2,EFR-510035,KNT-JQG,2022-09-08,field_observation,Uma Wessen,Horizon Field Research Cooperative,2022-09,Field Observation 28,SITE AND TIME\r\nField observation at KNT-JQG ...,BAT-0001037,DEV-KNTJQG-1,internal
3,EFR-510007,KNT-JQG,2025-02-15,field_observation,J. Varlan,Varlan Natural History Circle,2025-02,Field Observation 29,SITE AND TIME\r\nField observation at KNT-JQG ...,BAT-0001074,DEV-KNTJQG-1,public
4,EFR-510031,KNT-VLB,2020-07-22,field_observation,Jonah Bell,Bell Conservation Survey,2020-07,Field Observation 30,SITE AND TIME\r\nField observation at KNT-VLB ...,BAT-0001111,DEV-KNTVLB-2,internal


,document_id,site_code,document_date,document_type,author_name,author_organization,reporting_period,title,text,related_survey_id,related_device_reference,publication_status
27,EFR-510075,KNT-BME,2025-02-21,ecological_incident_report,Ravin Palen,Asterion Clinical Institute,2025-02,Ecological Incident Report 53,INCIDENT\r\nAn ordinary ecological incident wa...,BAT-0001962,DEV-KNTBME-1,restricted
15,EFR-510098,KNT-GGN,2021-02-18,cave_access_note,Jora Vela,Horizon Field Research Cooperative,2021-02,Cave Access Note 41,ACCESS CONTEXT\r\nRoutine access at KNT-GGN fo...,BAT-0001518,DEV-KNTGGN-1,internal
23,EFR-510012,KNT-VXR,2021-06-20,sensor_quality_report,NaN,Threshold Earth Observatory,2021-06,Sensor Quality Review 49,DEVICE\r\nQuality review for DEV-KNTVXR-2 at K...,BAT-0001814,DEV-KNTVXR-2,internal
17,EFR-510122,KNT-QWU,2019-12-05,cave_access_note,Jora Boras,Avelora Agricultural Resilience Agency,2019-12,Cave Access Note 43,ACCESS CONTEXT\r\nRoutine access at KNT-QWU fo...,BAT-0001592,DEV-KNTQWU-2,restricted
8,EFR-510037,KNT-YRG,2024-03-23,monitoring_report,Selene Aras,Threshold Earth Observatory,2024-03,Monthly Monitoring Report 34,"PERIOD\r\nMonitoring report for KNT-YRG, focus...",BAT-0001259,DEV-KNTYRG-1,restricted


Valeurs manquantes et doublons

In [32]:
display(pd.DataFrame({
    "valeurs_manquantes": df.isna().sum(),
    "pourcentage": (df.isna().mean() * 100).round(2)
}))
print("Doublons exacts :", df.duplicated().sum())

,valeurs_manquantes,pourcentage
document_id,0,0.00
site_code,0,0.00
document_date,0,0.00
document_type,0,0.00
author_name,1,3.33
author_organization,0,0.00
reporting_period,0,0.00
title,0,0.00
text,0,0.00
related_survey_id,0,0.00


Doublons exacts : 0


Formats et catégories

In [33]:
colonnes_categorielles = ['document_type', 'publication_status', 'site_code', 'author_organization','author_name']
for col in colonnes_categorielles:
    print(f"\n{col} :")
    print(df[col].value_counts(dropna=False).head(25))


document_type :
document_type
ecological_incident_report    6
field_observation             6
monitoring_report             6
cave_access_note              6
sensor_quality_report         6
Name: count, dtype: int64

publication_status :
publication_status
restricted    12
public        10
internal       8
Name: count, dtype: int64

site_code :
site_code
KNT-DMJ    3
KNT-JQG    2
KNT-VLB    2
KNT-CHP    2
KNT-YRG    2
KNT-WUQ    2
KNT-GGN    2
KNT-QWU    2
KNT-TLQ    2
KNT-UJJ    2
KNT-VXR    2
KNT-TEX    2
KNT-BME    2
KNT-KES    1
KNT-JAK    1
KNT-XAC    1
Name: count, dtype: int64

author_organization :
author_organization
Horizon Field Research Cooperative        6
Avelora Agricultural Resilience Agency    5
Threshold Earth Observatory               5
Pollinator Justice Network                4
Belen Agro-Industries                     4
Asterion Clinical Institute               2
Varlan Natural History Circle             1
Bell Conservation Survey                  1
Marot Ecologi

In [34]:
print("Dates manquantes ou invalides :",
      df['document_date'].isna().sum())

Dates manquantes ou invalides : 0


In [35]:
print("Périodes manquantes ou invalides :",
      df['reporting_period'].isna().sum())

Périodes manquantes ou invalides : 0


## Nettoyage des données

In [36]:
df_clean = df.copy()

uniformiser les catégories (espaces et majuscules/minuscules).

In [37]:
# Uniformisation des catégories
colonnes_categorielles = [
    'document_type',
    'publication_status',
    'site_code',
    'author_organization'
]

for col in colonnes_categorielles:
    df_clean[col] = (
        df_clean[col]
        .astype('string')
        .str.strip()
        .str.lower()
    )

# Nettoyage léger des colonnes textuelles
for col in ['title', 'text', 'author_name']:
    df_clean[col] = (
        df_clean[col]
        .astype('string')
        .str.strip()
    )

In [38]:
# Uniformiser les valeurs manquantes dans les colonnes textuelles
for col in df_clean.select_dtypes(
    include=['object', 'string']
).columns:

    df_clean[col] = (
        df_clean[col]
        .astype('string')
        .str.strip()
        .replace(
            r'(?i)^(na|n/a|null|none|nan)?$',
            pd.NA,
            regex=True
        )
    )

la date complète

In [39]:
df_clean['document_date'] = pd.to_datetime(
    df_clean['document_date'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_clean['reporting_period'] = pd.to_datetime(
    df_clean['reporting_period'],
    format='%Y-%m',
    errors='coerce'
)

# Vérification
print("Dates manquantes ou invalides :",
      df_clean['document_date'].isna().sum())

print("Périodes manquantes ou invalides :",
      df_clean['reporting_period'].isna().sum())

print(df_clean[['document_date', 'reporting_period']].dtypes)

Dates manquantes ou invalides : 0
Périodes manquantes ou invalides : 0
document_date       datetime64[us]
reporting_period    datetime64[us]
dtype: object


Vérifications

Analyser les valeurs manquantes

In [40]:
missing = pd.DataFrame({
    "nb_manquantes": df_clean.isna().sum(),
    "pourcentage": (df_clean.isna().mean() * 100).round(2)
})

display(missing[missing["nb_manquantes"] > 0])

,nb_manquantes,pourcentage
author_name,1,3.33


Vérifier les identifiants

In [41]:
# Vérifier l'unicité de document_id
doublons_id = df_clean.duplicated(
    subset=['document_id'],
    keep=False
)

print(
    "Documents avec identifiant dupliqué :",
    doublons_id.sum()
)

if doublons_id.any():
    display(df_clean.loc[doublons_id])

# Vérifier les identifiants manquants
print(
    "document_id manquants :",
    df_clean['document_id'].isna().sum()
)

print(
    "site_code manquants :",
    df_clean['site_code'].isna().sum()
)

Documents avec identifiant dupliqué : 0
document_id manquants : 0
site_code manquants : 0


Vérifier la cohérence des dates

In [42]:
# Début de la période de reporting
debut_periode = df_clean['reporting_period']

# Documents datés avant la période qu'ils couvrent
dates_suspectes = (
    df_clean['document_date'].notna()
    & debut_periode.notna()
    & (df_clean['document_date'] < debut_periode)
)

print(
    "Documents datés avant leur période :",
    dates_suspectes.sum()
)

if dates_suspectes.any():
    display(df_clean.loc[
        dates_suspectes,
        ['document_id', 'document_date', 'reporting_period']
    ])

Documents datés avant leur période : 0


In [43]:
# Nettoyage léger du texte sans modifier son contenu
for col in ['title', 'text']:
    df_clean[col] = (
        df_clean[col]
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )

# Vérification des textes manquants
print("Titres manquants :", df_clean['title'].isna().sum())
print("Textes manquants :", df_clean['text'].isna().sum())

# Vérification des textes vides ou très courts
longueurs = df_clean['text'].str.len()

print("Textes de moins de 50 caractères :",
      (longueurs < 50).sum())

display(
    df_clean.loc[
        longueurs < 50,
        ['document_id', 'title', 'text']
    ]
)

Titres manquants : 0
Textes manquants : 0
Textes de moins de 50 caractères : 0


,document_id,title,text


## Export des données nettoyées

In [44]:
print("===== BILAN FINAL DU NETTOYAGE =====")

print("Lignes initiales :", len(df))
print("Lignes finales :", len(df_clean))
print("Doublons exacts :", df_clean.duplicated().sum())

print("\nValeurs manquantes restantes :")
display(df_clean.isna().sum().to_frame("Nombre"))

print("\nTypes des colonnes :")
display(df_clean.dtypes.to_frame("Type"))

===== BILAN FINAL DU NETTOYAGE =====
Lignes initiales : 30
Lignes finales : 30
Doublons exacts : 0

Valeurs manquantes restantes :


,Nombre
document_id,0
site_code,0
document_date,0
document_type,0
author_name,1
author_organization,0
reporting_period,0
title,0
text,0
related_survey_id,0



Types des colonnes :


,Type
document_id,string
site_code,string
document_date,datetime64[us]
document_type,string
author_name,string
author_organization,string
reporting_period,datetime64[us]
title,string
text,string
related_survey_id,string


In [45]:
df_clean.to_csv(
    "environmental_field_reports_clean.csv",
    index=False
)
print("Fichier créé : environmental_field_reports.csv")

Fichier créé : environmental_field_reports.csv
